# AraBERTopic — Modern Standard Arabic Baseline

**Method:** Uses **AraBERT** (`aubmindlab/bert-base-arabertv02`) — a BERT model pretrained on Modern Standard Arabic (MSA), not dialect — as the sole contextual embedding, compressed with an autoencoder and fed into **CombinedTM** across a topic-count grid search.

**Why this method:** This notebook serves as a comparison baseline: how well does a strong *standard Arabic* model do on *dialectal* Tunisian text, without any TunBERT or dialect-specific embedding in the mix? This tells us how much value the TunBERT-based hybrid combinations in the other notebooks are actually adding over an off-the-shelf MSA model.

**Pipeline:** preprocess → AraBERT sentence embeddings (via SentenceTransformer wrapper) → UMAP-reduce to 128d → autoencoder-compress to 64d → CombinedTM, swept over topic counts 5-100.

## 0. Setup

In [ ]:
!pip install numpy==1.26.4 gensim==4.3.3 --force-reinstall --upgrade
!pip install transformers sentence-transformers umap-learn hdbscan scikit-learn nltk pandas openpyxl contextualized-topic-models tensorflow


## 1. Load the corpus

We start from the raw Tunisian dialect social media corpus. Each row is one post/message; we drop empty rows since they carry no signal for topic modeling.

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_excel("../data/TunTap_Corpus.xlsx")

# Drop rows with missing text
df = df.dropna(subset=["message"])

# Reset index
df = df.reset_index(drop=True)

# Extract just the text column
message = df["message"].tolist()


## 2. Preprocessing

Tunisian dialect on social media mixes Arabic script, Arabizi (Latin transliteration with digits standing in for Arabic letters, e.g. `3` for `ع`), French loanwords, elongated words for emphasis (`hhhhh`, `mriiiigla`), and noise (URLs, mentions, hashtags, emojis). Standard NLP preprocessing pipelines aren't built for this, so we apply dialect-specific cleaning:

1. Remove custom Tunisian stopwords (functional words with no topical meaning)
2. Normalize character elongation (`mriiiigla` → `mrigla`)
3. Convert Arabizi digits back to their Arabic-letter equivalent (`3` → `a`, `7` → `h`, `5` → `kh`, `9` → `k`, `2` → `a`)
4. Strip URLs, mentions, hashtags, emojis and non-alphanumeric symbols
5. Normalize Arabic letter variants (e.g. `إأآا` → `ا`) so the same word isn't split across multiple spellings
6. Remove stopwords a second time (some appear only after cleaning) and drop any resulting empty lines

In [ ]:
# Read stopwords.txt into a set
with open("../data/tunisian_stopwords.txt", "r", encoding="utf-8") as f:
    all_stopwords = set(line.strip() for line in f if line.strip())


In [ ]:
def remove_custom_stopwords(text, stopwords_set):
    tokens = text.split()
    filtered = [word for word in tokens if word not in stopwords_set]
    return " ".join(filtered)


In [ ]:
text = [remove_custom_stopwords(t, all_stopwords) for t in message]


In [ ]:
import re

def normalize_elongation(text):
    # Replace 2 or more repeated characters with 1
    return re.sub(r'(.)\1{1,}', r'\1', text)

def convert_tunisian_numbers(text):
    return (
        text.replace("3", "a")
            .replace("7", "h")
            .replace("5", "kh")
            .replace("9", "k")
            .replace("2", "a")
    )

def remove_numbers(text):
    # Remove all digits (0-9)
    return re.sub(r'\d+', '', text)

def clean_dual_script_text(text):
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Remove mentions and hashtags
    text = re.sub(r"@\w+|#\w+", "", text)

    # Remove emojis and symbols (non-alphanum + Arabic letters + space)
    text = re.sub(r"[^\u0621-\u063A\u0641-\u064A\w\s]", " ", text)

    # Normalize Arabic letters
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)

    # convert numbers like 3 → ع
    text = convert_tunisian_numbers(text)

    # Normalize elongation
    text = normalize_elongation(text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    text = remove_numbers(text)

    return text


In [ ]:
cleaned_texts = [clean_dual_script_text(t) for t in text]

In [ ]:
final_texts = [remove_custom_stopwords(t, all_stopwords) for t in cleaned_texts]

In [ ]:
final_texts = [line for line in final_texts if line.strip() != '']


Check how many documents survived preprocessing:

In [ ]:
print(len(final_texts))


17287


## 3. Topic coherence utilities

We evaluate topic quality with two complementary metrics:
- **C_V coherence** — measures how semantically related the top words of a topic are, based on word co-occurrence in a sliding window (via `gensim`)
- **NPMI** (Normalized Pointwise Mutual Information) — a simpler, more interpretable co-occurrence measure computed directly on document sets, less sensitive to corpus size than raw PMI

Both are computed from `final_texts`, split into tokens.

In [ ]:
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from itertools import combinations
from collections import Counter
import math

tokenized_docs = [doc.split() for doc in final_texts]
dictionary = Dictionary(tokenized_docs)

def compute_cv_score(topics_list, tokenized_docs, dictionary):
    cm = CoherenceModel(topics=topics_list, texts=tokenized_docs, dictionary=dictionary, coherence='c_v')
    return cm.get_coherence()

def compute_manual_npmi(topics_list, tokenized_docs):
    doc_sets = [set(d) for d in tokenized_docs]
    num_docs = len(doc_sets)
    word_doc_counts = Counter()
    for s in doc_sets:
        for w in s:
            word_doc_counts[w] += 1
    def topic_npmi(topic_words):
        vals = []
        for w1, w2 in combinations(topic_words, 2):
            p1 = word_doc_counts.get(w1, 0) / num_docs
            p2 = word_doc_counts.get(w2, 0) / num_docs
            co = sum(1 for s in doc_sets if w1 in s and w2 in s)
            p12 = co / num_docs
            if p12 > 0 and p1 > 0 and p2 > 0:
                pmi = math.log(p12 / (p1 * p2))
                npmi = pmi / (-math.log(p12))
                vals.append(npmi)
        return (sum(vals) / len(vals)) if vals else 0.0
    scores = [topic_npmi(topic) for topic in topics_list]
    return sum(scores) / len(scores), scores


## 4. AraBERT embeddings + bag-of-words

Unlike the TunBERT-based notebooks, AraBERT is loaded through `sentence-transformers` directly (as a general-purpose sentence encoder), and we build a fresh bag-of-words here restricted to the top 2000 features for this comparison.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np, re, pandas as pd

cleaned_docs = final_texts  # preprocessed Tunisian dialect texts

# AraBERT (Modern Standard Arabic, not dialect-specific)
arabert = SentenceTransformer("aubmindlab/bert-base-arabertv02")
embeddings = arabert.encode(cleaned_docs, show_progress_bar=True, convert_to_numpy=True)

# Bag-of-words
vectorizer = CountVectorizer(max_features=2000)
bow = vectorizer.fit_transform(cleaned_docs)
id2word = {i: w for i, w in enumerate(vectorizer.get_feature_names_out())}


## 5. Dimensionality reduction and compression

In [ ]:
import umap
umap_model = umap.UMAP(n_components=128, random_state=42)
arabert_reduced = umap_model.fit_transform(embeddings)


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

input_dim = arabert_reduced.shape[1]
input_layer = Input(shape=(input_dim,))
encoded = Dense(64, activation='relu')(input_layer)
decoded = Dense(input_dim, activation='linear')(encoded)
autoencoder = Model(input_layer, decoded)

autoencoder.compile(optimizer=Adam(), loss='mse')
autoencoder.fit(arabert_reduced, arabert_reduced, epochs=200, batch_size=32, verbose=1)

encoder = Model(input_layer, encoded)
compressed_embs = encoder.predict(arabert_reduced)


## 6. Build the CTM dataset

In [ ]:
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation

tp = TopicModelDataPreparation("aubmindlab/bert-base-arabertv02")

training_dataset = tp.fit(
    text_for_contextual=cleaned_docs,
    text_for_bow=cleaned_docs,
    custom_embeddings=compressed_embs
)

bow_size = len(tp.vocab)
contextual_size = compressed_embs.shape[1]  # 64


## 7. Grid search over topic counts

In [ ]:
# ===============================
# Grid search over topic counts
# ===============================
from contextualized_topic_models.models.ctm import CombinedTM

topic_numbers = [5, 6, 7, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 95, 100]
results = []

for n_topics in topic_numbers:
    print(f"\n🌀 Training CombinedTM with {n_topics} topics...")

    prodlda_arabert = CombinedTM(
        bow_size=bow_size,
        contextual_size=contextual_size,
        n_components=n_topics,
        num_epochs=5,
        batch_size=64,
    )

    prodlda_arabert.fit(training_dataset)

    topics = prodlda_arabert.get_topic_lists(20)
    topics_clean = [
        [w if isinstance(w, str) else w[0] for w in topic]
        for topic in topics
    ]

    texts_tokenized = [doc.split() for doc in cleaned_docs]
    dictionary = Dictionary(texts_tokenized)
    corpus = [dictionary.doc2bow(text) for text in texts_tokenized]
    cv_score = compute_cv_score(topics_clean, texts_tokenized, dictionary)
    avg_npmi, _ = compute_manual_npmi(topics_clean, texts_tokenized)

    print(f"✅ {n_topics} topics → CV: {cv_score:.3f}, NPMI: {avg_npmi:.3f}")

    results.append((n_topics, cv_score, avg_npmi))

# ===============================
# Display results as table
# ===============================
results_df = pd.DataFrame(results, columns=["n_topics", "CV", "NPMI"])
print("\n📊 Grid Search Results:")
print(results_df)



🌀 Training CombinedTM with 5 topics...

✅ 5 topics → CV: 0.463, NPMI: 0.553

🌀 Training CombinedTM with 6 topics...

✅ 6 topics → CV: 0.544, NPMI: 0.399

🌀 Training CombinedTM with 7 topics...

✅ 7 topics → CV: 0.482, NPMI: 0.603

🌀 Training CombinedTM with 10 topics...

✅ 10 topics → CV: 0.522, NPMI: 0.533

🌀 Training CombinedTM with 15 topics...

✅ 15 topics → CV: 0.525, NPMI: 0.490

🌀 Training CombinedTM with 20 topics...

✅ 20 topics → CV: 0.530, NPMI: 0.492

🌀 Training CombinedTM with 25 topics...

✅ 25 topics → CV: 0.482, NPMI: 0.488

🌀 Training CombinedTM with 30 topics...

✅ 30 topics → CV: 0.567, NPMI: 0.461

🌀 Training CombinedTM with 35 topics...

✅ 35 topics → CV: 0.522, NPMI: 0.540

🌀 Training CombinedTM with 40 topics...

✅ 40 topics → CV: 0.547, NPMI: 0.529

🌀 Training CombinedTM with 45 topics...

✅ 45 topics → CV: 0.573, NPMI: 0.549

🌀 Training CombinedTM with 50 topics...

✅ 50 topics → CV: 0.564, NPMI: 0.549

🌀 Training CombinedTM with 55 topics...

✅ 55 topics → CV

## Results

Best C_V coherence: **100 topics** (CV 0.577), also the best NPMI (0.701) — the highest scores of any *single* embedding source tested. Interesting given AraBERT isn't dialect-specific: at high topic counts it appears able to carve out narrow, coherent themes even on dialectal text, though this may partly reflect topics fragmenting into very small, easy-to-cohere clusters rather than genuinely broader distinct themes. Worth comparing manually against the actual topic words before concluding AraBERT "beats" the TunBERT hybrids.